# *Missa Veni sponsa Christi* — Ricerca dei soggetti

Questo notebook consente di cercare uno o più **soggetti** all'interno del mottetto *Veni sponsa Christi* e delle cinque parti della *Missa Veni sponsa Christi* di Giovanni Pierluigi da Palestrina. Le composizioni sono caricate direttamente dal database [CRIM Project](https://crimproject.org/).
Le partiture in formato PDF sono disponibili sul sito del progetto:<br>
[Veni sponsa Christi](https://crimproject.org/pieces/CRIM_Model_0019/)<br>
[Missa Veni sponsa Christi](https://crimproject.org/masses/CRIM_Mass_0019/)<br>


| Codice | Composizione |
|--------|-------------|
| `CRIM_Model_0019` | *Veni sponsa Christi* — mottetto (modello) |
| `CRIM_Mass_0019_1` | *Missa Veni sponsa Christi*: Kyrie |
| `CRIM_Mass_0019_2` | *Missa Veni sponsa Christi*: Gloria |
| `CRIM_Mass_0019_3` | *Missa Veni sponsa Christi*: Credo |
| `CRIM_Mass_0019_4` | *Missa Veni sponsa Christi*: Sanctus |
| `CRIM_Mass_0019_5` | *Missa Veni sponsa Christi*: Agnus Dei |

**Come usare il notebook:** esegui le celle nell'ordine seguendo le istruzioni di ogni sezione. Per salvare i risultati scegli *File → Download as → HTML*.

## 1. Avvio

Importa le librerie necessarie. Esegui questa cella prima di qualsiasi altra.

In [ ]:
import IPython
from IPython.display import HTML, display
from crim_intervals import importScore
import crim_intervals.visualizations as viz

audio_style = '<style>audio { margin-left: 0px; width: 960px; }</style>'
display(HTML(audio_style))
print('Ok')

## 2. Lista delle composizioni

La lista `piece_list` contiene i codici CRIM delle composizioni da analizzare. Per includere o escludere una parte della messa è sufficiente aggiungere o rimuovere il codice corrispondente.

In [ ]:
piece_list = [
    'CRIM_Model_0019',   # mottetto (modello)
    'CRIM_Mass_0019_1',  # Kyrie
    'CRIM_Mass_0019_2',  # Gloria
    'CRIM_Mass_0019_3',  # Credo
    'CRIM_Mass_0019_4',  # Sanctus
    'CRIM_Mass_0019_5',  # Agnus Dei
]
print(f'{len(piece_list)} composizioni in lista')

## 3. Parametri di ricerca

Tre parametri controllano il modo in cui vengono calcolati gli n-grammi melodici:

- **`n`** — lunghezza del soggetto espressa come numero di **intervalli** (es. `n = 4` corrisponde a quattro intervalli, ossia cinque note).
- **`kind`** — tipo di intervallo: `'d'` diatonico (es. terza = `3`, indipendentemente dall'alterazione); `'q'` di qualità (es. terza minore = `m3`, terza maggiore = `M3`).
- **`combineUnisons`** — se `True`, i ribattuti vengono raggruppati in un'unica nota; se `False` ogni ribattuto genera un intervallo di unisono (`'1'` quando **`kind`** = `'d'`).

In [ ]:
n              = 4      # numero di intervalli per soggetto
kind           = 'd'    # 'd' = diatonico  |  'q' = quality
combineUnisons = True   # True = collassa i ribattuti

print(f'n={n}, kind={kind!r}, combineUnisons={combineUnisons}')

## 4. Soggetti da cercare

Un **soggetto** è una sequenza di `n` intervalli scritti come stringhe. Convenzioni:

- segno negativo → moto discendente; nessun segno → moto ascendente; 
- il numero indica la distanza diatonica: `'2'` = seconda, `'3'` = terza, `'4'` = quarta, ecc.; `'m2'` = seconda minore, ecc. con `kind = 'q'`
- `'1'` rappresenta l'unisono (nota ribattuta); è rilevante solo con `combineUnisons = False`

È possibile inserire più soggetti nella stessa lista: tutti verranno cercati e visualizzati simultaneamente nelle mappe. I soggetti inclusi nella lista devono essere della stessa lunghezza.

**Esempio** — quattro intervalli corrispondenti all'incipit di *Veni sponsa*:

```python
soggetti = [('-3', '3', '2', '-2')]
```

### Configurazioni predefinite

Le celle seguenti contengono impostazioni già pronte per soggetti specifici del *Veni sponsa*. Eseguire **una sola** di queste celle per sostituire i parametri correnti con la configurazione desiderata, poi passare alla sezione 5.

In [ ]:
# I quattro soggetti del mottetto – quattro intervalli, senza ribattuti
n              = 4
kind           = 'd'
combineUnisons = True
soggetti = [
    ('-3', '3',  '2', '-2'),   # Veni sponsa Chri-
    ('-3', '2',  '2', '-2'),   # Accipe coro-
    ('-3', '3', '-2', '-2'),   # Quam tibi Dominus
    ('-2', '2',  '3', '-2'),   # Praeparavit in ae-
]
print(f'n={n}, combineUnisons={combineUnisons}  →  {len(soggetti)} soggetti')

In [ ]:
# Soggetto Quam tibi Dominus — tre intervalli, senza ribattuti
n              = 3
kind           = 'd'
combineUnisons = True
soggetti       = [('-3', '3', '-2')]
print(f'n={n}, combineUnisons={combineUnisons}  →  soggetti={soggetti}')

In [ ]:
# Soggetto Quam tibi Dominus — cinque intervalli, con i due ribattuti iniziali
n              = 5
kind           = 'd'
combineUnisons = False
soggetti       = [('1', '1', '-3', '3', '-2')]
print(f'n={n}, combineUnisons={combineUnisons}  →  soggetti={soggetti}')

In [ ]:
# Soggetto Accipe coronam — due frammenti separati: inizio e fine
n              = 4
kind           = 'd'
combineUnisons = True
soggetti       = [
    ('-3', '2',  '2', '-2'),   # frammento iniziale: accipe coro-
    ( '2', '2', '-2', '-4'),   # frammento finale: -cipe coronam
]
print(f'n={n}, combineUnisons={combineUnisons}  →  {len(soggetti)} soggetti')

## 5. Ricerca nelle mappe

La cella seguente carica ogni composizione della lista, calcola gli n-grammi melodici e visualizza due mappe per ciascuna composizione:

- **Mappa degli n-grams** — tutti gli n-grammi presenti nella composizione; i soggetti cercati appaiono evidenziati a colori.
- **Mappa delle entrate** — solo le occorrenze che compaiono come entrate, ossia dopo una pausa o all'inizio della composizione.

In ogni mappa ogni mattoncino corrisponde a un n-gramma e ogni riga a una voce; la posizione orizzontale indica dove l'n-gramma compare nella composizione (da 0 a 1 sull'intera durata). Passando il cursore sulle celle si leggono i dati dell'occorrenza. Sotto ogni coppia di mappe è presente la traccia audio.

In [ ]:
for mei_file in piece_list:
    url    = 'https://crimproject.org/mei/' + mei_file + '.mei'
    piece  = importScore(url)
    meta   = piece.metadata
    title  = meta['title']
    author = meta['composer']

    display(HTML(f'<hr><h3>{title}</h3><p><em>{author}</em></p>'))

    nr             = piece.notes(combineUnisons=combineUnisons)
    mel            = piece.melodic(df=nr, kind=kind, compound=True, unit=0, end=False)
    mel_ngrams     = piece.ngrams(df=mel, n=n)
    mel_ngrams_dur = piece.durations(df=mel, n=n, mask_df=mel_ngrams)

    display(HTML('<p><strong>Mappa degli n-grams</strong></p>'))
    display(viz.plot_ngrams_heatmap(mel_ngrams, mel_ngrams_dur, selected_patterns=soggetti, voices=[]))

    entries     = piece.entries(df=mel, n=n, thematic=True, anywhere=False)
    entries_dur = piece.durations(df=mel, n=n, mask_df=entries)

    display(HTML('<p><strong>Mappa delle entrate</strong></p>'))
    display(viz.plot_ngrams_heatmap(entries, entries_dur, selected_patterns=soggetti, voices=[]))

    display(IPython.display.Audio('https://crimproject.org/static/mp3/' + mei_file + '.mp3'))

## 6. Controllo in partitura

Per visualizzare un'occorrenza specifica direttamente in partitura, imposta:

- **`movement`** — indice della composizione nella lista (`0` = mottetto, `1`–`5` = parti della messa nell'ordine della lista)
- **`offset`** — valore *start* dell'occorrenza, leggibile passando il cursore sui mattoncini delle mappe

La cella calcola la battuta corrispondente all'offset indicato e visualizza un estratto di tre battute.

In [ ]:
movement = 0    # 0 = mottetto, 1–5 = parti della messa
offset   = 216  # valore 'start' dell'occorrenza (leggibile dalle mappe)

mei_file = piece_list[movement]
url      = 'https://crimproject.org/mei/' + mei_file + '.mei'
piece    = importScore(url)
nr       = piece.notes()
df       = piece.detailIndex(nr, offset=True)
measure  = df.loc[(slice(None), slice(None), offset), :].index.get_level_values('Measure').values[0]
print(f'Offset {offset}  →  battuta {int(measure)}')
piece.verovioPrintExample(int(measure), int(measure) + 2)